Importation des bibliothèques

In [91]:
import os
import random
# import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import random_split
from tqdm import tqdm # pour la progression 
import time #pour le temps de calcul
import copy #pour copier base_model

In [92]:
print(os.getcwd())  # donne le répertoire courant

d:\INRIA\MCDropout


Fixation du seed pour la reproductibilité

In [93]:
seed = 42
random.seed(seed)
# np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # si GPU dispo

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Configuration de base

Importation du modèle déjà entraîné par l'utilisateur

In [94]:
batch_size = 128 #on peut changer la taille mais 128 est bien pour éviter underfitting ou overfitting

transform = transforms.Compose([
    transforms.ToTensor(),                   #Convertir une image en tenseur, met les valeurs des pixels entre 0 et 1
    transforms.Normalize((0.5, 0.5, 0.5),    # Moyenne RGB
                         (0.5, 0.5, 0.5))])  # Écart-type RGB; pixels deviennent centrés autour de 0

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')


Définition du CNN de base (3 couches)

In [95]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # entrée 3 channels (RGB)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)  # réduit dimension par 2 à chaque fois
        
        # Calculer la taille en sortie avant les couches fully connected
        # CIFAR10 32x32 → après 3 poolings (x2) → 4x4
        self.fc1 = nn.Linear(64 * 4 * 4, 128) # après les convolutions on a un tenseur de taille 64*4*4, fc1 les réduit en 128 neurones
        self.fc2 = nn.Linear(128, 10)  # 10 classes CIFAR10

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        
        x = x.view(x.size(0), -1)  # aplatit en [batch_size, 64*4*4 = 1024]
        x = F.relu(self.fc1(x)) #ou fonction sigmoïde à la fin?
        x = self.fc2(x)
        return x



à entraîner (boucle d'entraînement) training loop + sauvegarde des poids à chaque epoch ; tester pour chaque epoch et voir où l'accuracy est la mieux

Masque pour le dropout

In [96]:
class CNN_MCdropout(nn.Module):
    def __init__(self, model, dico_layers=None):
        super().__init__()
        self.model = model
        self.dico_layers = dico_layers or {}

        # Remplacement des couches ciblées
        for name, layer in list(self.model._modules.items()):
            if isinstance(layer, nn.Conv2d) and name in self.dico_layers:
                p = self.dico_layers[name]
                self.model._modules[name] = nn.Sequential(layer, nn.Dropout2d(p), nn.ReLU())

            elif isinstance(layer, nn.Linear) and name in self.dico_layers:
                p = self.dico_layers[name]
                self.model._modules[name] = nn.Sequential(layer, nn.Dropout(p), nn.ReLU())
                
    # ici je mets le dropout en sortie, 
    def forward(self, x):
        return self.model.forward(x)
    

Fonction d'entraînement et test

In [97]:
val_ratio = 0.1  # 10% pour validation
train_size = int((1 - val_ratio) * len(trainset))
val_size   = len(trainset) - train_size

# on fixe tjrs les mêmes train et val set
train_subset, val_subset = random_split(trainset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))

trainloader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader   = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)


Fonction d'évaluation (avant entraînement)

In [98]:
def evaluate(model, dataloader, device):
    model.eval() # ou model.train() ?
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss/len(dataloader), correct/total

Fonction d'entraînement

In [99]:
def train_model(model, trainloader, valloader, device, epochs=20, save_path="best.pt"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0
    
    for epoch in range(epochs):#petit nombre d'epochs pour tester (environ 1 minutes pas epoch)
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Évaluer sur validation
        val_loss, val_acc = evaluate(model, valloader, device)
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {running_loss/len(trainloader):.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")
        
        # Sauvegarde si amélioration
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
    
    print("Finished Training")
    return model

In [100]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best.pt"

# # Demande à l'utilisateur quelles couches doivent subir le dropout : ou alors il amène sa liste
# user_layers = input(
#     "Sur quelles couches voulez-vous appliquer le MC Dropout ? "
#     "(choisissez parmi conv1, conv2, conv3, fc1, séparées par des virgules) : ")
# mc_layers = [layer.strip() for layer in user_layers.split(',') if layer.strip() in ['conv1','conv2','conv3','fc1']]

# Vérifie si les poids existent déjà
base_model = SimpleCNN()
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # même architecture que celle qui a sauvegardé
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = train_model(base_model, trainloader, valloader, device, epochs=20, save_path=save_path)
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # recharge les meilleurs poids

dico_layers = {
    "model": base_model,
    "layers": {
        "conv1": 0.2,
        "conv2": 0.2,
        "conv3": 0.2,
        "fc1": 0.2,
    }
} #nom : input_dico OU j'enlève le model et je mets le model en argument à part dans la classe générique 

# On fait une copie pour MC Dropout
base_model_mc = copy.deepcopy(base_model)

# Récupérer le dictionnaire des dropouts
mc_layers_dict = dico_layers["layers"]

# Copier le modèle pour le MC Dropout
base_model_dict = copy.deepcopy(dico_layers["model"])

# Créer le modèle MC Dropout générique correctement
model = CNN_MCdropout(base_model_dict, dico_layers=mc_layers_dict).to(device)

# Évaluation finale du modèle sans dropout
test_loss, test_acc = evaluate(model, testloader, device)
print(f"Final Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")


Chargement du modèle sauvegardé
Final Test Loss: 0.8186 - Test Acc: 0.7267


MC Dropout prédiction

In [101]:
def mc_predict_mean_probs(model, X, T=1000, verbose=True):
    model.train()  # activer dropout pendant l'inférence MC
    probs_list = []
    start_time = time.time() 
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            logits_t = model(X)
            probs_t = F.softmax(logits_t, dim=1)
            probs_list.append(probs_t.unsqueeze(0))
    elapsed = time.time() - start_time
    probs_mc = torch.cat(probs_list, dim=0)       
    model.eval()  # remettre le modèle en mode eval à la fin
    if verbose:
        print(f"Temps total: {elapsed:.2f} s  |  Temps moyen par passe: {elapsed/T:.4f} s")
    return probs_mc.mean(0), elapsed                        

je dois garder le même batch ; mettre des seeds pour que ce soit reproductible (fonction de dropout)

In [102]:
# Test MC Dropout sur un batch
X, Y = next(iter(testloader))
X, Y = X.to(device), Y.to(device)
probs, t1 = mc_predict_mean_probs(model, X, T=1000, verbose=True)
Y_hat = probs.argmax(1)

print("Classes vraies       :", Y.tolist())
print(f"Classes prédites   (t={t1:.2f}s): {Y_hat.tolist()}")

acc = (Y_hat == Y).float().mean().item()
print(f"MC Dropout Test Acc: {acc:.4f}")


100%|██████████| 1000/1000 [00:13<00:00, 73.93it/s]

Temps total: 13.54 s  |  Temps moyen par passe: 0.0135 s
Classes vraies       : [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9, 5, 2, 4, 0, 9, 6, 6, 5, 4, 5, 9, 2, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9, 7, 6, 9, 8, 0, 3, 8, 8, 7, 7, 4, 6, 7, 3, 6, 3, 6, 2, 1, 2, 3, 7, 2, 6, 8, 8, 0, 2, 9, 3, 3, 8, 8, 1, 1, 7, 2, 5, 2, 7, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 6, 2, 1, 3, 0, 4, 2, 7, 8, 3, 1, 2, 8, 0, 8, 3]
Classes prédites   (t=13.54s): [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 2, 4, 1, 4, 2, 4, 0, 9, 6, 3, 5, 2, 3, 9, 3, 4, 1, 9, 5, 4, 6, 3, 6, 0, 9, 3, 3, 7, 2, 9, 8, 2, 3, 8, 8, 5, 5, 5, 3, 7, 5, 6, 3, 6, 2, 1, 0, 3, 7, 0, 6, 8, 8, 8, 2, 2, 3, 3, 8, 8, 1, 1, 7, 2, 2, 2, 8, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 3, 2, 5, 6, 3, 1, 1, 2, 6, 8, 5, 6, 0, 2, 2, 9, 3, 0, 4, 3, 5, 8, 2, 1, 2, 8, 0, 8, 3]
MC Dropout Test Acc: 0.7500


rajouter en argument de generate_mc_outputs measure = variance (en gros on laisse à l'utilisateur le choix de la mesure d'incertitude)

In [103]:
def generate_mc_outputs(model, X, T=1000, metrics=["mc_estimate"], labels=None, verbose=True):
    model.train()
    outputs = []
    mean_probs = None 

    start_forward = time.time()
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            out = model(X)
            outputs.append(out.unsqueeze(0))
    elapsed_forward = time.time() - start_forward

    outputs = torch.cat(outputs, dim=0)  # [T, batch, num_classes]
    results = {}
    elapsed_metrics = {}

    # Calcul de la classe prédite initialement (premier passage)
    with torch.no_grad():
        first_logits = model(X)
        first_probs = torch.softmax(first_logits, dim=1)
        initial_pred = first_probs.argmax(dim=1)  # [batch]

    for metric in metrics:
        start_metric = time.time()

        if metric == "mc_estimate":
            all_probs = torch.softmax(outputs, dim=2)
            mean_probs = all_probs.mean(dim=0)
            results[metric] = mean_probs

        elif metric == "variance_predicted":
            # Variance des probabilités softmax de la classe prédite initialement
            all_probs = torch.softmax(outputs, dim=2)  # [T, batch, num_classes]
            # Pour chaque batch, on prend la colonne de la classe prédite initialement
            idx = initial_pred.unsqueeze(0).expand(T, -1)  # [T, batch]
            selected_probs = all_probs.gather(2, idx.unsqueeze(2)).squeeze(2)  # [T, batch]
            var_pred_class = selected_probs.var(dim=0)  # [batch]
            results["variance_predicted_mean"] = var_pred_class.mean().item()
            results["variance_predicted"] = var_pred_class

        # elif metric == "variance":
        #     # Variance des logits de la classe prédite initialement
        #     idx = initial_pred.unsqueeze(0).expand(T, -1)  # [T, batch]
        #     selected_logits = outputs.gather(2, idx.unsqueeze(2)).squeeze(2)  # [T, batch]
        #     var_fixed_class = selected_logits.var(dim=0)  # [batch]
        #     results["variance_mean"] = var_fixed_class.mean().item()
        #     results["variance"] = var_fixed_class

        elif metric == "variance":
            # Variance of softmax predictions across T
            all_probs = torch.softmax(outputs, dim=2)     # [T, batch, C]
            var_probs = all_probs.var(dim=0)              # [batch, C]
            results["variance"] = var_probs
            results["variance_mean"] = var_probs.mean().item()

        elif metric == "predictive_entropy_predicted":
            # Entropie prédictive pour la classe prédite initialement
            all_probs = torch.softmax(outputs, dim=2)  # [T, batch, num_classes]
            mean_probs = all_probs.mean(dim=0)         # [batch, num_classes]
            # On sélectionne la proba moyenne de la classe prédite initialement
            idx = initial_pred.unsqueeze(1)            # [batch, 1]
            selected_mean_probs = mean_probs.gather(1, idx).squeeze(1)  # [batch]
            # Entropie binaire pour la classe prédite (p*log(p) + (1-p)*log(1-p))
            entropies_pred = -(selected_mean_probs * (selected_mean_probs + 1e-12).log() +
                               (1 - selected_mean_probs) * ((1 - selected_mean_probs + 1e-12).log()))
            results["predictive_entropy_predicted_mean"] = entropies_pred.mean().item()
            results["predictive_entropy_predicted"] = entropies_pred

        elif metric == "predictive_entropy":
            # Entropie de la distribution moyenne (toutes classes)
            all_probs = torch.softmax(outputs, dim=2)
            mean_probs = all_probs.mean(dim=0)
            entropies = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)
            results["predictive_entropy_mean"] = entropies.mean().item()
            results["predictive_entropy"] = entropies

        elif metric == "relative_norm":
            if labels is None:
                raise ValueError("labels doivent être fournis pour relative_norm")
            all_probs = torch.softmax(outputs, dim=2)
            mean_probs = all_probs.mean(dim=0)
            labels_onehot = F.one_hot(labels, num_classes=mean_probs.size(1)).float()
            diff_norm = torch.norm(mean_probs - labels_onehot, dim=1)
            denom = torch.max(torch.norm(mean_probs, dim=1), torch.norm(labels_onehot, dim=1))
            relative_norm = diff_norm / (denom + 1e-12)
            results[metric + "_mean"] = relative_norm.mean().item()
            results[metric] = relative_norm

        else:
            raise ValueError(f"Métrique {metric} non reconnue")

        elapsed_metrics[metric] = time.time() - start_metric

    model.eval()

    if verbose:
        print(f"Temps forward pass: {elapsed_forward:.2f}s  |  Temps moyen par passe: {elapsed_forward/T:.4f}s")
        for m, t in elapsed_metrics.items():
            print(f"Temps calcul métrique '{m}': {t:.6f}s")

    return outputs, mean_probs, results, elapsed_forward, elapsed_metrics


le mc estimate n'est pas une mesure d'incertitude, sert pour construire une autre variable

Métriques

Métriques générique

In [104]:
# Choix des métriques par l'utilisateur
user_metrics = input(
    "Quelles métriques voulez-vous calculer ?\n"
    "Options disponibles : mc_estimate, variance_predicted, variance, predictive_entropy_predicted, predictive_entropy, relative_norm\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : "
)
user_metrics = [m.strip() for m in user_metrics.split(",")]

# Appel de la fonction
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(
    model, X, T=1000, metrics=user_metrics, labels=Y, verbose=False
)

print(f"\nListe des métriques choisies par l'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s\n")

# Affichage détaillé des résultats
for metric in user_metrics:
    print(f"--- Métrique choisie : {metric} ---")
    print(f"Temps de calcul : {elapsed_metrics[metric]:.6f} s")

    metric_result = metric_values[metric]

    if metric == "mc_estimate":
        print(f"Distribution de probabilités moyennes (par échantillon) :\n{metric_result}\n")
    
    elif metric == "variance":
        print(f"Variance moyenne (logits) : {metric_values['variance_mean']:.6f}")
        print(f"Variance par échantillon :\n{metric_result}\n")

    elif metric == "variance_predicted":
        print(f"Variance softmax moyenne : {metric_values['variance_predicted_mean']:.6f}")
        print(f"Variance softmax par échantillon :\n{metric_result}\n")

    elif metric == "predictive_entropy":
        print(f"Entropie prédictive moyenne : {metric_values['predictive_entropy_mean']:.6f}")
        print(f"Entropie prédictive par échantillon :\n{metric_result}\n")

    elif metric == "predictive_entropy_predicted":
        print(f"Entropie attendue moyenne : {metric_values['predictive_entropy_predicted_mean']:.6f}")
        print(f"Entropie attendue par échantillon :\n{metric_result}\n")

    elif metric == "relative_norm":
        print(f"Norme relative moyenne : {metric_values['relative_norm_mean']:.6f}")
        print(f"Norme relative par échantillon :\n{metric_result}\n")

    else:
        print(f"Résultat brut :\n")



Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance_predicted', 'variance', 'predictive_entropy_predicted', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 13.11 s

--- Métrique choisie : mc_estimate ---
Temps de calcul : 0.013695 s
Distribution de probabilités moyennes (par échantillon) :
tensor([[0.0648, 0.0366, 0.0540,  ..., 0.0119, 0.1578, 0.0300],
        [0.0561, 0.3409, 0.0136,  ..., 0.0045, 0.4923, 0.0228],
        [0.1767, 0.1018, 0.0597,  ..., 0.0362, 0.3508, 0.0583],
        ...,
        [0.3658, 0.0999, 0.2073,  ..., 0.0031, 0.1019, 0.0719],
        [0.2349, 0.0200, 0.1331,  ..., 0.0095, 0.2919, 0.0209],
        [0.0268, 0.0140, 0.0706,  ..., 0.0551, 0.0247, 0.0336]])

--- Métrique choisie : variance_predicted ---
Temps de calcul : 0.002110 s
Variance softmax moyenne : 0.097266
Variance softmax par échantillon :
tensor([0.1279, 0.1789, 0.1287, 0.1307, 0.1525, 0.1659, 0.1855, 0.1164, 0.1154,
        0.1619, 0.1375, 0.0430, 0.12

pour la fonction de confiance, dois-je faire 1/métrique choisie? ou ça marche qu'avec la variance?

oui si c'est dans R+ : 1/ ? exp(-...) si c'est pour avoir entre 0 et 1 ; quitte à normaliser après